In [1]:
import pandas as pd

# File names are swapped in this release — see 01_explore
train = pd.read_csv("../data/UNSW_NB15_testing-set.csv")
test = pd.read_csv("../data/UNSW_NB15_training-set.csv")

print(train.shape, test.shape)

(175341, 45) (82332, 45)


In [2]:
train = train.drop(columns='id')
test = test.drop(columns='id')

print(train.shape, test.shape)

(175341, 44) (82332, 44)


In [3]:
train_normal = train[train['label'] == 0]

print(train_normal.shape)

(56000, 44)


In [4]:
train_normal = train_normal.drop_duplicates()

print(train_normal.shape)

(51890, 44)


In [5]:
train_normal['proto'].value_counts()

proto
tcp     38165
udp     13000
arp       633
ospf       61
igmp       17
icmp       13
rtp         1
Name: count, dtype: int64

In [6]:
keep = ['tcp', 'udp', 'arp', 'ospf']

train_normal['proto'] = train_normal['proto'].where(train_normal['proto'].isin(keep), 'other')
test['proto'] = test['proto'].where(test['proto'].isin(keep), 'other')

print(train_normal['proto'].value_counts())
print(test['proto'].value_counts())

proto
tcp      38165
udp      13000
arp        633
ospf        61
other       31
Name: count, dtype: int64
proto
tcp      43095
udp      29418
other     8156
arp        987
ospf       676
Name: count, dtype: int64


## Preprocessing so far

- ID column dropped (just a row number, irrelevant to train on)

- Filtered out attacks into "train_normal" which is 56,000 rows (the model trains on normal traffic only)

- Removed duplicates leaving 51,890 rows (stop model overlearning)

- Compressed rare protocols into "other" (too low sample size for model to learn off (Kept tcp, udp, arp, ospf))

- Only 7 out of 133 protocols appear in normal traffic, essentially in this data rarer protocols indicate an attack (this wouldn't hold on a real network)

In [7]:
cat_cols = ['proto', 'service', 'state']

train_normal = pd.get_dummies(train_normal, columns=cat_cols, dtype=int)
test = pd.get_dummies(test, columns=cat_cols, dtype=int)

print(train_normal.shape, test.shape)

(51890, 65) (82332, 66)


In [ ]:
print("Only in test: ", set(test.columns) - set(train_normal.columns))
print("Only in train:", set(train_normal.columns) - set(test.columns))

Only in test:  {'service_irc', 'state_ACC', 'state_CLO', 'service_ssl', 'service_dhcp'}
Only in train: {'state_ECO', 'state_PAR', 'state_URN', 'state_no'}


In [ ]:
test = test.reindex(columns=train_normal.columns, fill_value=0)

print(train_normal.shape, test.shape)
print((train_normal.columns == test.columns).all()) 

(51890, 65) (82332, 65)
True


In [11]:
answer_cols = ['label', 'attack_cat']

X_train = train_normal.drop(columns=answer_cols)
X_test = test.drop(columns=answer_cols)

y_test = test['label']
attack_cat_test = test['attack_cat']

print(X_train.shape, X_test.shape, y_test.shape)

(51890, 63) (82332, 63) (82332,)


- One-hot encoded ['proto', 'service', 'state'] using get_dummies (each category becomes its own column, 1 if the row is that category, 0 if not)

- Reindexed test to match train’s columns (after get_dummies there was a mismatch (65 vs 66), because attacks in test used categories normal training traffic never did. Train defines the columns, since the model is built before new traffic arrives)

- Removed ['label', 'attack_cat'] from both, making X_train and X_test (so the model can't cheat by reading the answers, kept test answers for evaluation)